# Deep learning on generated credit-card datasets

Upload the prepared local bundle. It contains the original CSV, five generated-data variants plus the baseline on each of five saved splits, and the current code. The training matrix is 60 fits using an MLP classifier and a Transformer classifier.

Choose a GPU runtime in Colab. Training requires setting `RUN_TRAINING = True` in the dedicated cell. The delivered notebook has never been executed. See the bundled `colab/README.md` for recipes, evidence and resume behavior.

In [ ]:
import hashlib
import json
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload only credit_risk_colab.zip")
archive_path = next(iter(uploaded))
PROJECT = Path("/content/credit_risk_project")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        destination = (PROJECT / member.filename).resolve()
        if PROJECT.resolve() not in destination.parents:
            raise ValueError("Invalid archive path")
    archive.extractall(PROJECT)
manifest = json.loads((PROJECT / "COLAB_BUNDLE_MANIFEST.json").read_text())
for name, expected in manifest.items():
    if hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Bundle hash mismatch: {name}")
print(f"Verified {len(manifest)} bundled files")

Saving credit_risk_colab.zip to credit_risk_colab.zip
Verified 110 bundled files


In [ ]:
import os
import subprocess
import sys

os.chdir(PROJECT)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
subprocess.run([sys.executable, "-m", "pip", "check"])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'check'], returncode=1)

## Validate saved inputs

This cell checks the original CSV, all saved splits and training tables. It starts no model fitting and writes no experiment results.

In [ ]:
subprocess.run([sys.executable, "-m", "creditrisk.deep_learning", "--validate-only"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'creditrisk.deep_learning', '--validate-only'], returncode=0)

## Persistent output

Mount Drive and choose the output folder. Use a fresh folder for a new experiment. For an interrupted run, retain the same folder and set `RESUME = True` below. Only completed classifier jobs are reused. An interrupted classifier starts again at epoch one.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/credit_risk/deep_learning_comparison")

Mounted at /content/drive


## Explicit training launch

The default switch leaves training disabled. Set it to `True` when ready. The command requests CUDA, trains both neural classifiers across the saved datasets and splits, saves evidence and builds the neural report.

In [ ]:
RUN_TRAINING = True
RESUME = False

command = [
    sys.executable,
    "-m",
    "creditrisk.deep_learning",
    "--input",
    "results/credit_card",
    "--output",
    str(OUTPUT),
    "--device",
    "cuda",
    "--epochs",
    "100",
    "--patience",
    "15",
    "--batch-size",
    "512",
]
if RESUME:
    command.append("--resume")
print("Prepared command", command)
if RUN_TRAINING:
    subprocess.run(command, check=True)
else:
    print("Training disabled. Set RUN_TRAINING = True to launch.")

Prepared command ['/usr/bin/python3', '-m', 'creditrisk.deep_learning', '--input', 'results/credit_card', '--output', '/content/drive/MyDrive/credit_risk/deep_learning_comparison', '--device', 'cuda', '--epochs', '100', '--patience', '15', '--batch-size', '512']


## Read the completed comparison

The generated report and all prediction evidence stay in the configured Drive folder. Classical results already in the bundle describe the previous four classifiers. Neural results become available after this new training matrix completes.

In [ ]:
from IPython.display import Markdown, display

report = OUTPUT / "REPORT.md"
if report.exists():
    display(Markdown(report.read_text()))
else:
    print("A completed neural report is not available yet.")

# Deep learning on enlarged credit-card training datasets

Completed 60 neural classifier fits on the saved generation datasets.

The MLP uses hidden layers of 128 and 64 units with ReLU and dropout 0.2. The Transformer uses one token per scaled numeric or one-hot encoded component, width 32, four attention heads and two encoder layers with dropout 0.2. Mean pooling feeds the binary output head.

Both use unweighted binary cross-entropy, AdamW and the same recipe across datasets. Real validation AP selects the epoch. Real validation F1 selects the decision cutoff. The following scores are means over saved splits at that cutoff.

| Training method | Classifier | AP | ROC-AUC | F1 | Precision | Recall | Accuracy | Balanced accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| agss_diffusion_mlp | mlp_classifier | 0.5380 | 0.7756 | 0.5333 | 0.5173 | 0.5511 | 0.7873 | 0.7026 |
| agss_diffusion_mlp | transformer_classifier | 0.5484 | 0.7805 | 0.5383 | 0.5227 | 0.5563 | 0.7896 | 0.7060 |
| agss_diffusion_transformer | mlp_classifier | 0.5270 | 0.7626 | 0.5202 | 0.5042 | 0.5390 | 0.7806 | 0.6940 |
| agss_diffusion_transformer | transformer_classifier | 0.5282 | 0.7661 | 0.5263 | 0.5317 | 0.5236 | 0.7922 | 0.6960 |
| agss_interpolation | mlp_classifier | 0.5446 | 0.7722 | 0.5357 | 0.5133 | 0.5615 | 0.7853 | 0.7050 |
| agss_interpolation | transformer_classifier | 0.5451 | 0.7726 | 0.5327 | 0.5029 | 0.5681 | 0.7800 | 0.7041 |
| diffusion_mlp | mlp_classifier | 0.5408 | 0.7770 | 0.5373 | 0.5209 | 0.5564 | 0.7884 | 0.7052 |
| diffusion_mlp | transformer_classifier | 0.5445 | 0.7818 | 0.5382 | 0.5216 | 0.5563 | 0.7894 | 0.7058 |
| diffusion_transformer | mlp_classifier | 0.5270 | 0.7691 | 0.5267 | 0.5117 | 0.5444 | 0.7841 | 0.6982 |
| diffusion_transformer | transformer_classifier | 0.5346 | 0.7735 | 0.5332 | 0.5304 | 0.5375 | 0.7923 | 0.7010 |
| real | mlp_classifier | 0.5494 | 0.7786 | 0.5394 | 0.5151 | 0.5704 | 0.7849 | 0.7080 |
| real | transformer_classifier | 0.5507 | 0.7819 | 0.5378 | 0.5281 | 0.5501 | 0.7915 | 0.7050 |

The original-data row is the baseline for the same classifier. Paired differences subtract that baseline on the same split. Sample standard deviations and split counts appear in comparison_summary.csv. Repeated splits overlap, so these summaries describe this experiment.

Every score comes from a saved predictions.npz file in seed_SEED/METHOD/CLASSIFIER. It contains real labels, full-precision scores, the selected cutoff and original CSV row positions. history.csv records all validation APs and training losses. classifier.pt stores the selected weights. run.json records exact input hashes, training settings and runtime versions. Frozen source is in source/. The repository docs/METRICS.md defines the seven scores and confusion counts.
